In [0]:
import pandas as pd                                                     # Import pandas for data cleaning
import seaborn as sns                                                   # Import Seaborn for visualization
import matplotlib.pyplot as plt                                         # Import matplotlib library for visualization
import plotly.express as px                                             # Import plotly library for visualization
import plotly.graph_objects as go

from pyspark.sql import functions as F
from pyspark.sql.functions import col                                   # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev,count          # MathsFunctions
from pyspark.sql.functions import to_date,month,year,datediff           # DateFunctions
from pyspark.sql.functions import abs                                   # OtherFunctions

In [0]:
orders = spark.read.table("samplesuperstore.silverdata.orders")
returns = spark.read.table("samplesuperstore.silverdata.returns")
people = spark.read.table("samplesuperstore.silverdata.people")

In [0]:
# Column clustering into bins (Sales example)
from pyspark.sql.functions import col, when

orders_bins = orders.withColumn(
    "Sales_Bin",
    when(col("Sales") < 1000, "Low")
    .when((col("Sales") >= 1000) & (col("Sales") < 5000), "Medium")
    .otherwise("High")
)
display(orders_bins.groupBy("Sales_Bin").count())
orders_bins.toPandas()['Sales_Bin'].value_counts().plot(kind="bar")


In [0]:
# display(orders)

orders = spark.read.table("samplesuperstore.silverdata.orders")

# orders = orders.withColumn("OrderMonth", month(col("order_date")))
orders_trend = orders_with_month.groupBy("order_date").agg(F.sum("Sales").alias("MonthlySales"))
display(orders_trend)

pdf_trend = orders_trend.toPandas()

pdf_trend.plot(x="order_date", y="MonthlySales", kind="line", figsize=(10,6))


In [0]:
import matplotlib.pyplot as plt
import pyspark.sql.functions as F

# Frequency counts by Segment directly from orders
df_segmentation = orders.groupBy("Segment").count()

# Convert to Pandas for visualization
pdf_segmentation = df_segmentation.toPandas()

# Bar chart
pdf_segmentation.plot(
    x="Segment", 
    y="count", 
    kind="bar", 
    figsize=(8,5), 
    legend=False
)
plt.title("Customer Behavior Segmentation")
plt.ylabel("Number of Customers")
plt.xlabel("Segment")
plt.show()


In [0]:
from pyspark.sql.functions import col, datediff

# Derive ShippingDelay column
orders_delay = orders.withColumn(
    "ShippingDelay",
    datediff(col("Ship_Date"), col("Order_Date"))
)

# Convert to Pandas for visualization
pdf_delay = orders_delay.select("ShippingDelay").toPandas()

# Histogram of delays
import matplotlib.pyplot as plt
pdf_delay["ShippingDelay"].hist(bins=20)
plt.title("Shipping Delay Distribution")
plt.xlabel("Days")
plt.ylabel("Frequency")
plt.show()


In [0]:
# Convert to Pandas for full correlation matrix
pdf_orders = orders.select("Sales","Discount","Profit").toPandas()

import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(pdf_orders.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix: Sales, Discount, Profit")
plt.show()
